# Inference using LSDB-style API

In [1]:
import lsdb
import hyrax

In [2]:
from distributed import Client, LocalCluster

N_WORKERS, THREADS_PER_WORKER = 1, 1
cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 1,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:57304,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:57309,Total threads: 1
Dashboard: http://127.0.0.1:57310/status,Memory: 32.00 GiB
Nanny: tcp://127.0.0.1:57307,


## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [3]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [4]:
def create_meta_dict(nested_frame):
    meta_dict = {}
    cols = nested_frame._operation.meta.columns
    nested_cols = nested_frame._operation.meta.nested_columns

    non_nested_cols = [col for col in cols if col not in nested_cols]

    for col in non_nested_cols:
        meta_dict[col] = str(nested_frame[col].dtype)

    for col in nested_cols:
        meta_dict[col] = {}
        for ncols in nested_frame[col].columns:
            meta_dict[col][ncols] = str(nested_frame[f"{col}.{ncols}"].dtype)

    return meta_dict

In [5]:
h = hyrax.Hyrax()
h.config["model"]["name"] = "HyraxLoopback"
h.config["data_loader"]["batch_size"] = 64

h.config["data_request"] = {
    "infer": {
        "data": {
            "dataset_class": "DataframeDataset",
            "data_location": create_meta_dict(tess_filtered_nans),
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj", "lightcurve_sap_flux"],
        }
    }
}

h.set_config("infer.model_weights_file", "none")

## Use a Dask Worker Plugin to provide a model for use with inference

In [6]:
import uuid

import torch
import numpy as np
import pandas as pd
from distributed import WorkerPlugin, get_worker

from hyrax.models.model_registry import fetch_model_class
from hyrax.models.model_utils import load_model_weights
from hyrax.plugin_utils import load_prepare_inputs
from hyrax.pytorch_ignite import _auto_model, create_process_func, dist_data_loader, setup_dataset

PRED_META = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)


def predict(df, process_fn, dataset, config):
    with torch.no_grad():
        # Note that `dataset` is a Hyrax DataProvider that is passed to `dist_data_loader`
        # It is not the bare dataset class.
        dataset.prepped_datasets["data"].data = df

        #! Be aware, this is Pytorch-ignite.
        data_loader = dist_data_loader(dataset, config)

        for b in data_loader:
            # The first `None` parameter is a placeholder for the pytorch ignite engine
            print(b["data"]["ra_obj"])
            model_out = process_fn(None, b)
            print(model_out)
            # This can be made more general, by default the return value will be
            # a bare tensor, but it could also be a dictionary of tensors.
            z = model_out.to("cpu").numpy() if len(b) else np.zeros(0, np.float32)

    df["pred"] = z
    return df
    # return pd.DataFrame({"pred": z}, index=df.index)


class TorchModelPlugin(WorkerPlugin):
    def __init__(self, name, config):
        self.name = name  # plugins are keyed by name on the worker
        self.config = config
        self.model = None
        self.dataset = None

    def setup(self, worker):
        model_cls = fetch_model_class(self.config)
        model = model_cls(self.config, None)

        #! Load the model weights using config value
        # load_model_weights(self.config, model, 'infer')

        #! Load the `prepare_inputs` function and attach it to the model
        # model.prepare_inputs = load_prepare_inputs(<input_directory>)

        # This is superfluous but could be used for diagnostics
        model.load_token = uuid.uuid4().int >> 65

        model.eval()
        self.model = _auto_model(model)

        # Build the function that is called that processes data
        self.process_fn = create_process_func("infer_batch", "mps", self.model, self.config)

        # setup the dataset class
        datasets = setup_dataset(self.config, shuffle=False)
        self.dataset = datasets["infer"]

    def teardown(self, worker):
        self.model = None


def predict_with_plugin(df, plugin_name):
    return predict(
        df,
        get_worker().plugins[plugin_name].process_fn,
        get_worker().plugins[plugin_name].dataset,
        get_worker().plugins[plugin_name].config,
    )

In [7]:
client.register_plugin(TorchModelPlugin("test-model", h.config))

# we want to combine the output of the model with the existing metadata
met = tess_filtered_nans._operation.meta
met["pred"] = PRED_META["pred"]

tess_pred = tess_filtered_nans.map_partitions(predict_with_plugin, plugin_name="test-model", meta=met)

tess_pred.compute()

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

2026-09-29 16:01:55,017 - distributed.worker - ERROR - Compute Failed
Key:       ('drop_nans-bb8fd72287e42296dcc0aa866a92d794', 3)
State:     executing
Task:  <Task ('drop_nans-bb8fd72287e42296dcc0aa866a92d794', 3) wrapped_func(...)>
Exception: 'ValueError("Metadata mismatch found in `wrapped_func`.\\n\\nPartition type: `nested_pandas.nestedframe.core.NestedFrame`\\n+--------+-------+----------+\\n| Column | Found | Expected |\\n+--------+-------+----------+\\n| \'pred\' | -     | float32  |\\n+--------+-------+----------+")'
Traceback: '  File "/Users/drew/opt/miniconda3/envs/hyrax/lib/python3.13/site-packages/lsdb/operations/lsdb_ops.py", line 38, in wrapper\n    return run_and_verify_meta(func, meta, *args, **kwargs)\n  File "/Users/drew/opt/miniconda3/envs/hyrax/lib/python3.13/site-packages/lsdb/operations/lsdb_ops.py", line 29, in run_and_verify_meta\n    check_meta(result, meta, funcname=funcname(func))\n    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/drew/o

ValueError: Metadata mismatch found in `wrapped_func`.

Partition type: `nested_pandas.nestedframe.core.NestedFrame`
+--------+-------+----------+
| Column | Found | Expected |
+--------+-------+----------+
| 'pred' | -     | float32  |
+--------+-------+----------+

2026-09-29 16:02:10,378 - distributed.worker - ERROR - Compute Failed
Key:       ('drop_nans-bb8fd72287e42296dcc0aa866a92d794', 1)
State:     executing
Task:  <Task ('drop_nans-bb8fd72287e42296dcc0aa866a92d794', 1) wrapped_func(...)>
Exception: 'ValueError("Metadata mismatch found in `wrapped_func`.\\n\\nPartition type: `nested_pandas.nestedframe.core.NestedFrame`\\n+--------+-------+----------+\\n| Column | Found | Expected |\\n+--------+-------+----------+\\n| \'pred\' | -     | float32  |\\n+--------+-------+----------+")'
Traceback: '  File "/Users/drew/opt/miniconda3/envs/hyrax/lib/python3.13/site-packages/lsdb/operations/lsdb_ops.py", line 38, in wrapper\n    return run_and_verify_meta(func, meta, *args, **kwargs)\n  File "/Users/drew/opt/miniconda3/envs/hyrax/lib/python3.13/site-packages/lsdb/operations/lsdb_ops.py", line 29, in run_and_verify_meta\n    check_meta(result, meta, funcname=funcname(func))\n    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/drew/o

In [ ]:
client.close()
cluster.close()